# Milestone 3a - Task 2: Corruption Classifier, Specialist Autoencoders & Hard Routing Pipeline

This notebook implements the complete Task 2 workflow for **Generative AI Restoration Studio**:
1. **Component 1: Corruption Classifier** — 4-way CNN (`Clean`, `Salt & Pepper`, `Gaussian Blur`, `Rectangular Occlusion`) tuned via Optuna (15 trials) and trained with multi-metric evaluation (Accuracy, Confusion Matrix, Macro F1).
2. **Component 2: Specialist Autoencoders** — 3 distinct autoencoders trained exclusively on 100% single-corruption streams with strict batch-level anti-contamination assertions.
3. **Component 3: Hard-Routing Pipeline** — Classifier-driven routing with exact **Identity Bypass** for clean inputs (PSNR = inf, SSIM = 1.0) and specialist dispatch.
4. **Comparative Benchmark** — Rigorous 3-way evaluation comparing **Universal Autoencoder** (Task 1) vs **Oracle Specialists** (Ceiling) vs **Predicted Hard-Routed Pipeline** on the full 3,669 test set across all corruption types and severity tiers.
5. **Visualizations & ONNX Export** — Side-by-side restoration panels and ONNX deployment export with numerical equivalence checks.

## Step 1: Environment Setup & GPU Verification

In [ ]:
# Verify GPU/CPU runtime
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU (Inference & export supported; training/benchmark will run on CPU).")

## Step 2: Google Drive Mount, Repository Sync & Dependencies Installation
Mounts Google Drive, clones/pulls the latest repository, installs all Colab dependencies (including `mlflow`, `optuna`, `onnxscript`), caches images to fast local Colab disk (`/content/local_data/`), and configures the MLflow SQLite database.

In [ ]:
import os
import sys
import shutil

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully at /content/drive")
except ImportError:
    print("Running outside Google Colab environment.")

# 2. Clone or update repository
REPO_URL = 'https://github.com/UsmanBari/genai-restoration-studio.git'
REPO_DIR = '/content/genai-restoration-studio'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install required Colab dependencies (mlflow, optuna, onnxscript, etc.)
!pip install -q -r requirements-colab.txt
print("Dependencies installed successfully.")

# 4. Fast bulk copy of images to local Colab NVMe disk (Task 1 proven pattern)
DRIVE_IMAGES_DIR = '/content/drive/MyDrive/GenAI-A1/raw/OxfordPet/images_128x128'
LOCAL_DATA_DIR = '/content/local_data/OxfordPet'
LOCAL_IMAGES_DIR = os.path.join(LOCAL_DATA_DIR, 'images_128x128')
os.makedirs(LOCAL_IMAGES_DIR, exist_ok=True)

existing_local_count = len([f for f in os.listdir(LOCAL_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

if existing_local_count >= 7349:
    print(f"⚡ Local NVMe cache already populated ({existing_local_count} images in {LOCAL_IMAGES_DIR}). Skipping copy.")
elif os.path.exists(DRIVE_IMAGES_DIR):
    print(f"🚀 Caching 7,349 images from Google Drive to local Colab NVMe disk: {LOCAL_IMAGES_DIR}...")
    !cp -r {DRIVE_IMAGES_DIR}/* {LOCAL_IMAGES_DIR}/
    cached_count = len([f for f in os.listdir(LOCAL_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"✅ Successfully cached {cached_count} images to local path: {LOCAL_IMAGES_DIR}")
else:
    print(f"Active images directory set to fallback: {DRIVE_IMAGES_DIR}")
    LOCAL_IMAGES_DIR = DRIVE_IMAGES_DIR

# 5. Configure MLflow SQLite tracking backend on Drive
import mlflow
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
MLFLOW_DB_PATH = '/content/drive/MyDrive/GenAI-A1/mlruns_task2.db'
os.makedirs(os.path.dirname(MLFLOW_DB_PATH), exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
print(f"MLflow tracking URI configured: sqlite:///{MLFLOW_DB_PATH}")

## Step 3: Module Imports & Verify Manifests

In [ ]:
from data.oxford_pet import OxfordPetDataset, get_oxford_dataloaders
from data.corruptions import CORRUPTION_NAMES, NAME_TO_CLASS
from models.classifiers import CorruptionClassifier
from models.autoencoders import UniversalAutoencoder
from models.hard_router import HardRoutingRestorationPipeline
from training.trainer_classifier import train_classifier_full, evaluate_classifier
from training.optuna_classifier import run_classifier_optuna_study
from training.trainer_specialist import train_specialist_full, evaluate_specialist
from evaluation.benchmark_hard_routing import run_hard_routing_benchmark
from models.onnx_export import export_to_onnx, verify_onnx_numerical_equivalence

MANIFEST_DIR = 'configs/manifests'
DRIVE_CHECKPOINTS_DIR = '/content/drive/MyDrive/GenAI-A1/checkpoints/task2'
os.makedirs(DRIVE_CHECKPOINTS_DIR, exist_ok=True)
print("Task 2 modules loaded successfully.")

## Step 4: Component 1 — Corruption Classifier
### 4a. Optuna Hyperparameter Search (15 Trials, MedianPruner)
Searches `lr`, `batch_size`, `base_channels`, and `dropout_rate` with balanced runtime corruption sampling.

In [ ]:
RUN_CLASSIFIER_OPTUNA = True

if RUN_CLASSIFIER_OPTUNA:
    optuna_results = run_classifier_optuna_study(
        manifest_dir=MANIFEST_DIR,
        images_dir=LOCAL_IMAGES_DIR,
        n_trials=15,
        trial_epochs=3,
        study_name="task2_classifier_optuna_study"
    )
    best_classifier_params = optuna_results['best_params']
    print("Best Classifier Params:", best_classifier_params)
else:
    best_classifier_params = {
        'lr': 0.001,
        'batch_size': 32,
        'base_channels': 32,
        'dropout_rate': 0.2
    }

### 4b. Full 15-Epoch Training of Winning Classifier
Trains the classifier on the full training manifest with balanced runtime corruptions (25% clean, 25% S&P, 25% blur, 25% occlusion), saving `best_classifier.pth`.

In [ ]:
train_loader_cls, val_loader_cls, test_loader_cls = get_oxford_dataloaders(
    manifest_dir=MANIFEST_DIR,
    images_dir=LOCAL_IMAGES_DIR,
    batch_size=best_classifier_params.get('batch_size', 32),
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

classifier_model = CorruptionClassifier(
    in_channels=3,
    num_classes=4,
    base_channels=best_classifier_params.get('base_channels', 32),
    dropout_rate=best_classifier_params.get('dropout_rate', 0.2)
)

cls_train_res = train_classifier_full(
    model=classifier_model,
    train_loader=train_loader_cls,
    val_loader=val_loader_cls,
    epochs=15,
    lr=best_classifier_params.get('lr', 0.001),
    checkpoint_dir=DRIVE_CHECKPOINTS_DIR,
    experiment_name="Task2_Classifier_Training"
)

# Evaluate on Test Set
criterion_cls = torch.nn.CrossEntropyLoss()
test_cls_metrics = evaluate_classifier(
    model=classifier_model,
    dataloader=test_loader_cls,
    criterion=criterion_cls,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nTest Accuracy: {test_cls_metrics['accuracy']:.2f}%")
print(f"Macro F1: {test_cls_metrics['macro_f1']:.4f}")
print("Per-Class Recall:", test_cls_metrics['per_class_recall'])
print("Confusion Matrix:\n", test_cls_metrics['confusion_matrix'])

## Step 5: Component 2 — Specialist Autoencoders Training
We train 3 dedicated specialist autoencoders (reusing the gated skip + 8x8x96 bottleneck architecture):
1. **Salt & Pepper Specialist** (40 epochs on 100% S&P stream)
2. **Gaussian Blur Specialist** (40 epochs on 100% Blur stream)
3. **Rectangular Occlusion Specialist** (40 epochs on 100% Occlusion stream)

*Note*: Every batch is verified via strict batch-level assertions to guarantee zero cross-contamination.

In [ ]:
specialist_configs = [
    ('salt_and_pepper', 40, 2.33e-4, 0.90),
    ('gaussian_blur', 40, 2.33e-4, 0.90),
    ('rectangular_occlusion', 40, 2.33e-4, 0.90)
]

specialist_models = {}

for c_name, epochs, lr, alpha in specialist_configs:
    print(f"\n========================================")
    print(f"Training Specialist: {c_name}")
    print(f"========================================")
    
    train_loader_spec, val_loader_spec, _ = get_oxford_dataloaders(
        manifest_dir=MANIFEST_DIR,
        images_dir=LOCAL_IMAGES_DIR,
        batch_size=16,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        corruption_mode=c_name
    )
    
    spec_model = UniversalAutoencoder(
        in_channels=3,
        out_channels=3,
        base_channels=64,
        bottleneck_dim=96,
        dropout_rate=0.0
    )
    
    spec_res = train_specialist_full(
        target_corruption=c_name,
        model=spec_model,
        train_loader=train_loader_spec,
        val_loader=val_loader_spec,
        epochs=epochs,
        lr=lr,
        alpha=alpha,
        checkpoint_dir=DRIVE_CHECKPOINTS_DIR,
        experiment_name="Task2_Specialists"
    )
    
    # Load best checkpoint
    ckpt = torch.load(spec_res['best_checkpoint_path'], map_location='cpu')
    spec_model.load_state_dict(ckpt['model_state_dict'])
    spec_model.eval()
    specialist_models[c_name] = spec_model
    print(f"Specialist '{c_name}' loaded from best checkpoint.")

## Step 6: Component 3 — Hard Routing Pipeline Assembly
Assemble the hard-routing pipeline combining the trained `CorruptionClassifier`, the 3 trained specialists, and the zero-loss **Identity Bypass** on clean images.

In [ ]:
# Load best classifier checkpoint
best_cls_ckpt_path = os.path.join(DRIVE_CHECKPOINTS_DIR, "best_classifier.pth")
cls_ckpt = torch.load(best_cls_ckpt_path, map_location='cpu')
classifier_model.load_state_dict(cls_ckpt['model_state_dict'])
classifier_model.eval()

# Construct Hard Routing Pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"
hard_routing_pipeline = HardRoutingRestorationPipeline(
    classifier=classifier_model,
    specialists=specialist_models,
    device=device
)
print("Hard Routing Pipeline assembled successfully on device:", device)

## Step 7: 3-Way Comparative Benchmark on Test Set (Self-Contained / Standalone)
Evaluates the three paradigms on the full 3,669 test set:
1. **Task 1 Universal Autoencoder** (Single model restoring all corruptions)
2. **Oracle Hard-Routed Specialists** (Ground-truth routing: theoretical ceiling)
3. **Predicted Hard-Routed Pipeline** (Classifier-driven routing + Clean identity bypass)

*Note*: This cell is 100% self-contained and automatically restores all models from Google Drive checkpoints if run in a fresh or restarted kernel.

In [ ]:
import os
import torch
from models.autoencoders import UniversalAutoencoder
from models.classifiers import CorruptionClassifier
from models.hard_router import HardRoutingRestorationPipeline
from data.oxford_pet import get_oxford_dataloaders
from evaluation.benchmark_hard_routing import run_hard_routing_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active benchmark compute device: {device}")

# --------------------------------------------------------
# 1. Load Task 1 Universal Autoencoder from Checkpoint
# --------------------------------------------------------
universal_model = UniversalAutoencoder(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    bottleneck_dim=96,
    dropout_rate=0.0
)

candidate_u_paths = [
    '/content/drive/MyDrive/GenAI-A1/checkpoints/task1_universal_best.pth',
    '/content/drive/MyDrive/GenAI-A1/checkpoints/best_universal.pth',
    '/content/drive/MyDrive/GenAI-A1/checkpoints/task1/best_universal.pth',
    'checkpoints/task1_universal_best.pth',
    'checkpoints/task1/best_universal.pth'
]
loaded_u_path = next((p for p in candidate_u_paths if os.path.isfile(p)), None)
if loaded_u_path is None:
    raise FileNotFoundError("Task 1 checkpoint not found in candidate paths:\n" + "\n".join(f"  - {p}" for p in candidate_u_paths))

u_ckpt = torch.load(loaded_u_path, map_location='cpu')
universal_model.load_state_dict(u_ckpt['model_state_dict'])
universal_model.to(device).eval()
print(f"Loaded Task 1 Universal Autoencoder from: {loaded_u_path}")

# --------------------------------------------------------
# 2. Load Task 2 Classifier from Checkpoint (if needed)
# --------------------------------------------------------
DRIVE_CHECKPOINTS_DIR = '/content/drive/MyDrive/GenAI-A1/checkpoints/task2'
cls_ckpt_path = os.path.join(DRIVE_CHECKPOINTS_DIR, "best_classifier.pth")
if not os.path.isfile(cls_ckpt_path):
    cls_ckpt_path = 'checkpoints/task2/best_classifier.pth'

cls_ckpt = torch.load(cls_ckpt_path, map_location='cpu')
classifier_model = CorruptionClassifier(
    in_channels=3,
    num_classes=4,
    base_channels=cls_ckpt.get('base_channels', 32),
    dropout_rate=cls_ckpt.get('dropout_rate', 0.2)
)
classifier_model.load_state_dict(cls_ckpt['model_state_dict'])
classifier_model.to(device).eval()
print(f"Loaded Classifier from: {cls_ckpt_path} (Val Acc: {cls_ckpt.get('val_accuracy', 0):.2f}%)")

# --------------------------------------------------------
# 3. Load Task 2 Specialists from Checkpoints (if needed)
# --------------------------------------------------------
specialist_models = {}
for c_name in ['salt_and_pepper', 'gaussian_blur', 'rectangular_occlusion']:
    spec_path = os.path.join(DRIVE_CHECKPOINTS_DIR, f"best_specialist_{c_name}.pth")
    if not os.path.isfile(spec_path):
        spec_path = f"checkpoints/task2/best_specialist_{c_name}.pth"
    
    s_ckpt = torch.load(spec_path, map_location='cpu')
    spec_m = UniversalAutoencoder(
        in_channels=3,
        out_channels=3,
        base_channels=s_ckpt.get('base_channels', 64),
        bottleneck_dim=s_ckpt.get('bottleneck_dim', 96),
        dropout_rate=0.0
    )
    spec_m.load_state_dict(s_ckpt['model_state_dict'])
    spec_m.to(device).eval()
    specialist_models[c_name] = spec_m
    print(f"Loaded Specialist '{c_name}' (PSNR: {s_ckpt.get('val_psnr', 0):.2f}dB, SSIM: {s_ckpt.get('val_ssim', 0):.4f})")

# --------------------------------------------------------
# 4. Assemble Hard Routing Pipeline
# --------------------------------------------------------
hard_routing_pipeline = HardRoutingRestorationPipeline(
    classifier=classifier_model,
    specialists=specialist_models,
    device=device
).to(device).eval()

# --------------------------------------------------------
# 5. Execute 3-Way Comparative Benchmark
# --------------------------------------------------------
MANIFEST_DIR = 'configs/manifests'
LOCAL_IMAGES_DIR = '/content/local_data/OxfordPet/images_128x128'
if not os.path.exists(LOCAL_IMAGES_DIR):
    LOCAL_IMAGES_DIR = '/content/drive/MyDrive/GenAI-A1/raw/OxfordPet/images_128x128'

_, _, test_loader_full = get_oxford_dataloaders(
    manifest_dir=MANIFEST_DIR,
    images_dir=LOCAL_IMAGES_DIR,
    batch_size=32,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

benchmark_output_json = '/content/drive/MyDrive/GenAI-A1/benchmarks/task2_hard_routing_benchmark_results.json'
summary = run_hard_routing_benchmark(
    universal_model=universal_model,
    hard_router=hard_routing_pipeline,
    test_loader=test_loader_full,
    output_json_path=benchmark_output_json,
    device=device
)

## Step 8: Multi-Panel Restoration Visualizations
Generates side-by-side visual panels comparing **Input**, **Universal Recon**, **Hard-Routed Recon**, and **Ground Truth** for all 4 corruption types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sample representative test images (one for each corruption class)
test_ds = OxfordPetDataset(manifest_path=MANIFEST_DIR, images_dir=LOCAL_IMAGES_DIR, split='test')

samples_by_class = {}
for item in test_ds:
    lbl = item['label']
    if lbl not in samples_by_class:
        samples_by_class[lbl] = item
    if len(samples_by_class) == 4:
        break

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ["Corrupted Input", "Universal (Task 1)", "Hard-Routed (Task 2)", "Ground Truth"]

device = "cuda" if torch.cuda.is_available() else "cpu"
universal_model.to(device).eval()
hard_routing_pipeline.to(device).eval()

for row_idx, (lbl, sample) in enumerate(sorted(samples_by_class.items())):
    c_name = CORRUPTION_NAMES[lbl].replace('_', ' ').title()
    corrupted_t = sample['corrupted'].unsqueeze(0).to(device)
    clean_t = sample['clean'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        u_recon = torch.clamp(universal_model(corrupted_t), 0.0, 1.0)
        h_recon, pred_c, _ = hard_routing_pipeline.route_predicted(corrupted_t)
    
    in_np = sample['corrupted'].permute(1, 2, 0).cpu().numpy()
    u_np = u_recon[0].permute(1, 2, 0).cpu().numpy()
    h_np = h_recon[0].permute(1, 2, 0).cpu().numpy()
    gt_np = sample['clean'].permute(1, 2, 0).cpu().numpy()
    
    imgs = [in_np, u_np, h_np, gt_np]
    pred_name = CORRUPTION_NAMES[pred_c.item()].replace('_', ' ').title()
    
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        ax.imshow(imgs[col_idx])
        ax.axis('off')
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=13, fontweight='bold')
    
    axes[row_idx, 0].text(-15, 64, f"{c_name}\n(Pred: {pred_name})", 
                          fontsize=11, fontweight='bold', va='center', ha='right')

plt.tight_layout()
vis_path = '/content/drive/MyDrive/GenAI-A1/visualizations/task2_hard_routing_comparison.png'
os.makedirs(os.path.dirname(vis_path), exist_ok=True)
plt.savefig(vis_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Visualization saved to: {vis_path}")

## Step 9: ONNX Export & Equivalence Verification
Export all models to ONNX and verify numerical parity.

In [ ]:
!pip install -q onnxscript

EXPORT_DIR = '/content/drive/MyDrive/GenAI-A1/exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Export Classifier
cls_onnx_path = os.path.join(EXPORT_DIR, 'task2_classifier.onnx')
dummy_input = torch.randn(1, 3, 128, 128)
torch.onnx.export(
    classifier_model.cpu(),
    dummy_input,
    cls_onnx_path,
    opset_version=14,
    input_names=['input_image'],
    output_names=['logits'],
    dynamic_axes={'input_image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}}
)
print(f"Classifier exported to ONNX ({os.path.getsize(cls_onnx_path) / 1e6:.2f} MB).")

# 2. Export Specialists
for c_name, spec_m in specialist_models.items():
    spec_onnx_path = os.path.join(EXPORT_DIR, f'task2_specialist_{c_name}.onnx')
    export_to_onnx(spec_m.cpu(), spec_onnx_path, opset_version=14)
    res = verify_onnx_numerical_equivalence(spec_m.cpu(), spec_onnx_path)
    print(f"Specialist '{c_name}' ONNX exported ({res['onnx_size_mb']:.2f} MB), Max Diff: {res['max_abs_diff']:.2e}, Parity: {res['equivalent']}")